[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/stammler/simframe/HEAD?labpath=examples%2F7_signal_handling.ipynb)

**This notebook teaches how to:**  
interact with a Simframe simulation from outside of its own process.

# 7. Signal Handling

Simframe has the capability to interact with it from outside its process. This can be used to trigger the writing of dump and/or data files of a running simulation or to stop the simulation entirely. Typical use cases can be to inspect the current state of a simulation before the next snapshot is written, or to write dump files and terminate a simulation before the time limit of a SLURM job it met.

In this notebook we set up a simple mock simulation without any integration instruction, that is simply advancing the integration variable.

In [1]:
import numpy as np
from simframe import Frame
from simframe import Integrator
from simframe import writers

In [2]:
sim = Frame()

In [3]:
sim.writer = writers.hdf5writer(datadir="7_data", overwrite=True)

In [4]:
sim.addintegrationvariable("x", 0.)
sim.x.updater = lambda sim: 0.15
sim.x.snapshots = np.arange(0., 6., 1.)

In [5]:
sim.integrator = Integrator(sim.x)

In [6]:
sim.run()

Writing file 7_data/data0000.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0001.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0002.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0003.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0004.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0005.hdf5
Writing dump file 7_data/frame.dmp
Execution time: 0:00:00


## File Signals

### Dump Files

Now we want to trigger the writing of dump files immeadiately before they would be schedules by the simulation. This can be done by simypl creating a file called `DUMP` (capital letters for case sensitive operating systems) in the data directory of the simulation.

In this example we a creating a systole that is touching this file as soon as $\large x = \pi$, to simulate the creation of the file by the user. We add a flag to the simulation frame to make sure the file is only created once.

In [7]:
sim.writeflag = True
sim.writefile = "DUMP"

In [8]:
def systole(sim):
    if sim.writeflag and sim.x > np.pi:
        file = sim.writer.datadir / sim.writefile
        file.touch()
        sim.writeflag = False
sim.updater.systole = systole        

We reset the integration variable and run the simulation again.

In [9]:
sim.x = 0.

In [10]:
sim.run()

Writing file 7_data/data0000.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0001.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0002.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0003.hdf5
Writing dump file 7_data/frame.dmp
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0004.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0005.hdf5
Writing dump file 7_data/frame.dmp
Execution time: 0:00:00


As can be seen, after the output `0003`, there are two consecutive dump files written, because we triggered the event at $\large x = \pi$.

**Note:** This only works if the Frame has a writer with a data directory assigned, since Simframe needs to know where to look for the trigger files and how to write dump or output files. After the event has been triggered, the triggering file has been automatically deleted.

In [11]:
(sim.writer.datadir / sim.writefile).is_file()

False

### Output Files

In the next example we want Simframe to write unscheduled output **and** dump files. This can be achieved by creating the file `WRITE` in the data directory of the writer.

To demonstrate this, we change the file name, activate the flag, and reset the integration variable.

In [12]:
sim.writeflag = True
sim.writefile = "WRITE"
sim.x = 0.

Now we can run the simulation again.

In [13]:
sim.run()

Writing file 7_data/data0000.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0001.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0002.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0003.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/__OUTPUT__
Writing dump file 7_data/frame.dmp
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0004.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0005.hdf5
Writing dump file 7_data/frame.dmp
Execution time: 0:00:00


As can be seen, after the output `0003` an unscheduled output file `__OUTPUT__` has been written to the data directory. Aftwards two dump files were created. One was created by the writer after writing the output file `__OUTPUT__`, since the writing of dump files at outputs was activated (`sim.writer.dumping = True`). The second one was triggered by the `WRITE` file to make sure that a dump file is always written by this event, even if the writer has dumping deactived.

**Note:** The `__OUTPUT__` has the same file format as defined by the writer and will always be forcefully overwritten, even if `sim.writer.overwrite = False`.

### Stop the Simulation

In this example, we want to trigger the writing of data **and** dump files **and** stop the simulation. This can be achieved by creating the file `STOP` in the data directory of the writer.

To demonstrate this, we change the file name, activate the flag, and reset the integration variable.

In [14]:
sim.writeflag = True
sim.writefile = "STOP"
sim.x = 0.

Now we can run the simulation again. However, since this will trigger a `SystemExit` exception we need to catch it to keep this notebook going.

In [15]:
try:
    sim.run()
except SystemExit:
    print("-----> SystemExit")

Writing file 7_data/data0000.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0001.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0002.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/data0003.hdf5
Writing dump file 7_data/frame.dmp
Writing file 7_data/__OUTPUT__
Writing dump file 7_data/frame.dmp
Writing dump file 7_data/frame.dmp
-----> SystemExit


As can be seen, after the output `0003` the additional output file `__OUTPUT__` and twice the dump file has been written as with the `WRITE` file previously, then the `SystemExit` exception has been raised and the simulation has stopped.

## System Signals

In addition to file signals in the simulation's data directory, system signals sent to the simulation's process can trigger the writing ouf data and dump files and stop the simulation. By default Simframe is listening for the termination signal `SIGTERM (15)`.

In [16]:
import signal

In [17]:
try:
    signal.raise_signal(signal.SIGTERM)
except SystemExit:
    print("-----> SystemExit")

Signal detected: SIGTERM (15)
Writing file 7_data/__OUTPUT__
Writing dump file 7_data/frame.dmp
Writing dump file 7_data/frame.dmp
-----> SystemExit


Here the `SIGTERM` signal triggered the writing out data and output files and exited the simulation with a `SystemExit` exception. This even worked without the simulation running.

In this example `signal.raise_signal()` was sending the signal to its own process. But it is also possible to send a signal to another process if the process ID `pid` is known. This can be done with `os` module:

```python
import signal
import os

pid = 12345

os.kill(pid, signal.SIGTERM)
```

## SLURM Jobs

This functionality of Simframe can be used to trigger the writing out dump and output files just before a SLURM job would hit its time limit. This is especially useful to restart a simulation right from the last state and not from the last scheduled snapshot.

A typical SLURM script could look as follows:

```bash
#!/bin/bash 
#SBATCH ---job-name my_job
# .
# .
# .
#SBATCH --time=24:00:00
#SBATCH --signal=B:SIGTERM@300
# .
# .
# .
#SBATCH --export=NONE 
#SBATCH --get-user-env

python start.py
```

This job will run for a maximum of 24 hours. 300 seconds before the time limit is reached the `SIGTERM` signal will be sent to the process, which will trigger Simframe to write output and dump files and terminate the simulation. Note that the signal may be not sent exactly at the requested time and that – depending on the size of the simulation – the writing of the files can take some time. It is therefore recommended to add some safety margin.